# This is done in google collab because the local device had computational limitations

In [ ]:
# mount the drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:

cd /content/drive/MyDrive/Colab Notebooks/

In [ ]:
from transformers import CLIPProcessor, CLIPModel
import torch
from PIL import Image

In [ ]:
model_name = "openai/clip-vit-large-patch14"

model = CLIPModel.from_pretrained(model_name)
processor = CLIPProcessor.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

In [ ]:
def get_text_embedding(text):
    # Added truncation and max_length to handle long text inputs
    inputs = processor(text=[text], return_tensors="pt", padding=True, truncation=True, max_length=77).to(device)

    with torch.no_grad():
        outputs = model.get_text_features(**inputs)
        # Extract the tensor from the output object
        emb = outputs.pooler_output if hasattr(outputs, 'pooler_output') else outputs

    # Normalize the embedding
    emb = emb / emb.norm(dim=-1, keepdim=True)
    return emb.squeeze().cpu()

In [ ]:
def get_image_embedding(image_path):
    image = Image.open(image_path).convert("RGB")

    inputs = processor(images=image, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.get_image_features(**inputs)
        # Extract the tensor from the output object
        emb = outputs.pooler_output if hasattr(outputs, 'pooler_output') else outputs

    emb = emb / emb.norm(dim=-1, keepdim=True)
    return emb.squeeze().cpu()

In [ ]:
import os
def list_image_files(directory):
    image_extensions = ['.jpg', '.jpeg', '.png', '.gif', '.bmp', '.tiff', '.webp']
    image_files = []
    for root, _, files in os.walk(directory):
        for file in files:
            if any(file.lower().endswith(ext) for ext in image_extensions):
                image_files.append(os.path.join(root, file))
    return image_files

In [ ]:
drive_path = '/content/drive/MyDrive/Colab Notebooks/images/'
all_image_files = list_image_files(drive_path)

if all_image_files:
    print("Found image files:")
    for img_file in all_image_files:
        print(img_file)
else:
    print("No image files found in Google Drive.")

In [ ]:
if all_image_files:
    image_to_use = all_image_files[0]  # Using the first found image as an example
    print(f"Using image: {image_to_use}")
    image_embedding = get_image_embedding(image_to_use)
    print(f"Length of image embedding: {len(image_embedding)}")
else:
    print("No image files found to process.")

In [ ]:
import json
data = None
with open ('./impression_and_findings.json','r') as f:
  data = json.load(f)
print(data)

In [ ]:
from collections import Counter
uids = [item['id'] for item in data]
# Count occurrences of each UID
uid_counts = Counter(uids)
# Identify repeats
repeated_uids = {uid: count for uid, count in uid_counts.items() if count > 1}
# Results summary
print(f"Total entries in JSON: {len(data)}")
print(f"Total unique UIDs: {len(uid_counts)}")
print(f"Number of repeated UIDs: {len(repeated_uids)}")
if repeated_uids:
    print("\nRepeated UIDs and their counts:")
    for uid, count in repeated_uids.items():
        print(f"UID {uid}: {count} times")
else:
    print("\nSuccess: No repeated UIDs found! dataset is unique.")

In [ ]:
import numpy as np

results_to_save = []
output_file_path = '/content/drive/MyDrive/Colab Notebooks/embeddings_results.json'

print(f"Processing {len(data)} entries...")
for entry in data:
    # Get text embedding
    text_data = entry['text']
    text_embedding = get_text_embedding(text_data)

    # Get image embedding
    image_name = entry['image']
    image_path = os.path.join(drive_path, image_name)

    try:
        image_embedding = get_image_embedding(image_path)

        # Calculate final embedding (weighted average)
        # final_embedding = 0.6 * image_embedding + 0.4 * text_embedding

        # Store as lists for JSON serialization
        results_to_save.append({
            "id": entry['id'],
            "text_embedding": text_embedding.tolist(),
            "image_embedding": image_embedding.tolist(),
            # "final_embedding": final_embedding.tolist()
        })

        print(f'processed file {entry['id']}')
    except Exception as e:
        print(f"Error processing ID {entry['id']}: {e}")

# Save to JSON file
with open(output_file_path, 'w') as f:
    json.dump(results_to_save, f)

print(f"Successfully saved {len(results_to_save)} entries to {output_file_path}")

In [ ]:
check = []
for entry in results_to_save:
  check.append(entry['id'])

from collections import Counter
# find all id with count more than one
duplicates = [item for item, count in Counter(check).items() if count > 1]
print(duplicates)